# Estimating weights in the HogDSim agent-state model

This notebook runs the repository's swIAVs EMULSION variant and demonstrates
ABC-SMC estimation of within-partition (`trans`), schedule-mediated between-partition
(`trans_betwn_herds`), neighbour-partition (`trans_neighbor`), and auxiliary-route
(`trans_aux`) transmission parameters. It covers the paper's pen-count observation
scenario only; the simulator still includes the whole farm.

**The default observations are generated by the model and labelled synthetic.**
The original notebook's precomputed synthetic observation file is not included. This short example
checks the software workflow; it does not reproduce the published study or
establish parameter identifiability. Related article:
[10.1016/j.atech.2025.101652](https://doi.org/10.1016/j.atech.2025.101652).

## Before running

Use Python 3.12 and install `requirements-lock.txt` as described in the README.
Start Jupyter from this repository and select that environment's Python kernel.
Choose **Restart Kernel and Run All Cells**. No Redis server or cluster is needed.
The default example can take several minutes on a laptop. Each run saves its
configuration, observation provenance, SQLite history, posterior particles,
and plots in a unique `results/estimation-*` directory.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
from pathlib import Path
import random
from uuid import uuid4

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyabc
from pyabc.sampler import SingleCoreSampler

from swine_inference import MODELS, PARAMETER_MAPS, distance, load_observations, simulate

print(f"EMULSION {version('emulsion')}; pyABC {version('pyabc')}")
print("Available models:", ", ".join(MODELS))

## Configure the example

The defaults use 8 weekly steps, 12 particles, and at most 2 ABC populations to
keep the demonstration small. These settings are not research recommendations.
The article (sections 2.3.1 and 2.3.3) used 100 accepted particles, up to 5
populations, adaptive Manhattan distance with MedianEpsilon and threshold 0.001.
It selected a synthetic trajectory closest to the median of 100 simulations; its
swIAVs application used a 170-week horizon. Those expensive settings are not
executed by this quick example. A study also needs convergence and predictive checks.

Set `OBSERVATIONS_CSV` to a real CSV path to use your own observations. It must
contain `week,nb_I`, with one row for each week from 0 through
`SIMULATION_STEPS`. `nb_I` counts **pen partitions containing I-state agents (0 to 12)**,
not individual agents. The notebook rejects missing/duplicate weeks and invalid
counts; it never replaces supplied observations with synthetic data.

In [ ]:
SEED = 2026
MODEL_VARIANT = "swiavs"
MODEL = MODELS[MODEL_VARIANT]
PARAMETERS = PARAMETER_MAPS[MODEL_VARIANT]
SIMULATION_STEPS = 8
SYNTHETIC_REPLICATES = 1  # use 100 to exercise the paper's closest-to-median procedure
POPULATION_SIZE = 12
MAX_POPULATIONS = 2
MIN_EPSILON = 0.001
DISTANCE_METHOD = "adaptive_l1"  # or "unscaled_l1" for an explicit simpler comparison
MAX_SIMULATIONS = 200  # pyABC checks this stopping limit between populations
OBSERVATIONS_CSV = None  # for example, Path("data/my_observations.csv")

# All synthetic generating values lie inside the example's prior support.
SYNTHETIC_PARAMETERS = {"trans": 17.01, "trans_betwn_herds": 9.87, "trans_neighbor": 9.87, "trans_aux": 0.1}
INITIAL_PARAMETERS = {"trans": 10.0, "trans_betwn_herds": 5.0, "trans_neighbor": 5.0, "trans_aux": 0.5}
PRIOR_UPPER = {"trans": 20.0, "trans_betwn_herds": 15.0, "trans_neighbor": 15.0, "trans_aux": 1.0}

if not isinstance(SYNTHETIC_REPLICATES, int) or SYNTHETIC_REPLICATES < 1:
    raise ValueError("SYNTHETIC_REPLICATES must be a positive integer")

np.random.seed(SEED)
random.seed(SEED)
run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
RUN_DIRECTORY = Path("results") / ("estimation-" + run_name)
RUN_DIRECTORY.mkdir(parents=True)
RUN_DIRECTORY = RUN_DIRECTORY.resolve()
weeks = np.arange(SIMULATION_STEPS + 1)
print(f"Results: {RUN_DIRECTORY}")

## Load observations or generate a synthetic example

Synthetic observations use the article's generating transmission values. With
`SYNTHETIC_REPLICATES=1`, the quick example uses a single separate trajectory.
With multiple replicates, it selects the actual run closest to the componentwise
median, using Euclidean distance as in the original notebook. The median itself
is never used as a fabricated state-count sequence. Saved seeds initialize the
random streams, but EMULSION's object-identity-based agent hashes mean they do not
guarantee bit-identical trajectories; actual observations are saved for reuse.
These generated counts are not empirical state measurements or a recovered original dataset.

In [ ]:
if OBSERVATIONS_CSV is None:
    candidate_seeds = [SEED + index for index in range(SYNTHETIC_REPLICATES)]
    candidates = np.stack([
        simulate(SYNTHETIC_PARAMETERS, steps=SIMULATION_STEPS, seed=seed, model_variant=MODEL_VARIANT)
        for seed in candidate_seeds
    ])
    median_trajectory = np.median(candidates, axis=0)
    selected_index = int(np.argmin(np.sum((candidates - median_trajectory) ** 2, axis=1)))
    observed = candidates[selected_index].copy()
    pd.DataFrame(candidates, columns=[f"week_{week}" for week in weeks]).to_csv(
        RUN_DIRECTORY / "synthetic_candidates.csv", index=False
    )
    reference_parameters = SYNTHETIC_PARAMETERS
    observation_label = "Synthetic observations"
    provenance = {"type": "synthetic", "parameters": SYNTHETIC_PARAMETERS,
                  "candidate_seeds": candidate_seeds, "selected_candidate": selected_index,
                  "selection": "closest actual trajectory to componentwise median (Euclidean)"}
else:
    observation_path = Path(OBSERVATIONS_CSV).expanduser().resolve(strict=True)
    observed = load_observations(observation_path, steps=SIMULATION_STEPS)
    reference_parameters = None
    observation_label = "User observations"
    provenance = {"type": "user_observations", "path": str(observation_path)}

pd.DataFrame({"week": weeks, "nb_I": observed}).to_csv(
    RUN_DIRECTORY / "observations.csv", index=False
)
configuration = {
    "model": MODEL.name, "variant": MODEL_VARIANT, "steps": SIMULATION_STEPS, "seed": SEED,
    "population_size": POPULATION_SIZE, "max_populations": MAX_POPULATIONS,
    "max_simulations": MAX_SIMULATIONS, "distance": DISTANCE_METHOD, "minimum_epsilon": MIN_EPSILON,
    "prior_upper": PRIOR_UPPER, "observations": provenance,
    "versions": {name: version(name) for name in ("emulsion", "pyabc", "numpy", "pandas")},
}
(RUN_DIRECTORY / "configuration.json").write_text(json.dumps(configuration, indent=2))
print(observation_label, observed)

## Run a trajectory at an initial guess

The adapter uses the actual YAML, Python add-on, and movement CSV shipped in this
repository. It gives every simulation a temporary directory, checks that all
weekly outputs exist, and removes those temporary files afterward. Failures
raise an exception rather than returning invented or zero-filled trajectories.
The initial pen count is derived from the initial I-state agents because
the summary process has not yet run at week 0.

In [ ]:
initial_trajectory = simulate(INITIAL_PARAMETERS, steps=SIMULATION_STEPS, seed=SEED + 1000, model_variant=MODEL_VARIANT)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weeks, observed, "o-", label=observation_label)
ax.plot(weeks, initial_trajectory, "--", label="One simulation at initial parameters")
ax.set(xlabel="Week", ylabel="Pen partitions containing I-state agents", ylim=(-0.5, 12.5))
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIRECTORY / "initial_trajectory.png", dpi=150)
plt.show()

## Define priors, the simulator, and the distance

The four uniform prior bounds follow section 2.3.1 of the article: 0..20,
0..15, 0..15, and 0..1 per week. The default adaptive Manhattan distance uses
`AdaptivePNormDistance(p=1)` with pyABC's PCMAD scaling (as in the original
notebook); scale weights are logged. Constant summary entries may receive zero
weight. `unscaled_l1` is an explicit optional comparison and is not the paper's
adaptive distance. The swIAVs transmission equations are preserved; known
differences from the article are listed in `docs/reproducibility.md`.

The serial sampler avoids the original unused Redis placeholder and uncontrolled
use of every CPU. Each model evaluation draws a fresh simulation seed. The
adapter restores the caller's NumPy and Python random states after simulation,
so EMULSION does not reset ABC's sampling stream.

In [ ]:
parameter_prior = pyabc.Distribution(**{
    name: pyabc.RV("uniform", 0.0, PRIOR_UPPER[name]) for name in PARAMETERS
})

def model(parameters):
    return {"infected_pens": simulate(parameters, steps=SIMULATION_STEPS, model_variant=MODEL_VARIANT)}

if DISTANCE_METHOD == "adaptive_l1":
    distance_function = pyabc.AdaptivePNormDistance(
        p=1, scale_function=pyabc.distance.pcmad,
        scale_log_file=str(RUN_DIRECTORY / "distance_scales.json"),
    )
elif DISTANCE_METHOD == "unscaled_l1":
    distance_function = distance
else:
    raise ValueError("DISTANCE_METHOD must be adaptive_l1 or unscaled_l1")

abc = pyabc.ABCSMC(
    model, parameter_prior, distance_function, population_size=POPULATION_SIZE,
    eps=pyabc.MedianEpsilon(), sampler=SingleCoreSampler(),
)
db_path = "sqlite:///" + str(RUN_DIRECTORY / "abc.sqlite")
abc_id = abc.new(db_path, {"infected_pens": observed})
print(f"ABC run ID: {abc_id}")

## Run ABC-SMC

pyABC reports calibration, accepted particles, acceptance rates, and stopping
conditions. Output is visible so failures and unexpectedly expensive runs can be
diagnosed. Increasing the trajectory length, particles, or populations increases
runtime. The simulation-count stopping limit is checked between populations and
is not a strict per-evaluation cap.

In [ ]:
history = abc.run(
    minimum_epsilon=MIN_EPSILON,
    max_nr_populations=MAX_POPULATIONS,
    max_total_nr_simulations=MAX_SIMULATIONS,
)
if history.max_t < 0:
    raise RuntimeError("ABC completed without an accepted population")
posterior, weights = history.get_distribution(m=0, t=history.max_t)
posterior_output = posterior.copy()
posterior_output["weight"] = weights
posterior_output.to_csv(RUN_DIRECTORY / "posterior.csv", index=False)
print(history.get_all_populations())
print(f"Completed {history.max_t + 1} population(s); saved {len(posterior)} particles.")

## Inspect the approximate posterior

Intervals and marginal densities below describe the small ABC demonstration.
Their appearance is not evidence of convergence or recovery of the generating
parameters. Reference lines identify the generating parameters only when using
synthetic observations; no ground truth is assigned to user observations.

In [ ]:
axes = pyabc.visualization.plot_credible_intervals(
    history, par_names=list(PARAMETERS), levels=[0.95], show_mean=True,
    refval=reference_parameters, size=(10, 3),
)
plt.gcf().tight_layout()
plt.gcf().savefig(RUN_DIRECTORY / "credible_intervals.png", dpi=150)
plt.show()

In [ ]:
axes = pyabc.visualization.plot_kde_matrix(
    posterior, weights,
    limits={name: (0.0, PRIOR_UPPER[name]) for name in PARAMETERS},
    refval=reference_parameters,
    names={"trans": "Within-partition", "trans_betwn_herds": "Schedule-mediated", "trans_neighbor": "Neighbours", "trans_aux": "Auxiliary"},
    height=2.5,
)
plt.gcf().tight_layout()
plt.gcf().savefig(RUN_DIRECTORY / "posterior_densities.png", dpi=150)
plt.show()

## One simulation at the posterior mean

This comparison is a single stochastic trajectory, not a posterior predictive
interval or a validation of the transition-rule system. A mathematical evaluation needs
multiple posterior predictive simulations and an observation model appropriate
to the actual sampling process.

In [ ]:
posterior_mean = {
    name: float(np.average(posterior[name], weights=weights)) for name in PARAMETERS
}
posterior_trajectory = simulate(posterior_mean, steps=SIMULATION_STEPS, seed=SEED + 2000, model_variant=MODEL_VARIANT)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weeks, observed, "o-", label=observation_label)
ax.plot(weeks, posterior_trajectory, "--", label="One simulation at posterior mean")
ax.set(xlabel="Week", ylabel="Pen partitions containing I-state agents", ylim=(-0.5, 12.5))
ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIRECTORY / "posterior_mean_trajectory.png", dpi=150)
plt.show()
print("Saved results to", RUN_DIRECTORY)

## Further computational evaluation

Preserve observation provenance, justify the prior bounds and distance, increase
the simulation horizon and inference budget, and check convergence, parameter
identifiability, and predictive performance. Changing the application of the rule set requires a separate review of state
definitions, timing, connectivity, and weights. Successful execution alone does
not validate those rules or their agreement with the article.